In [ ]:
import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from PIL import Image
import os
import json
import torch
import numpy as np
from PIL import Image
from datasets import Dataset, DatasetDict
from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
DATASET_DIR = "./sroie-datasetv2/SROIE2019"
MODEL_ID = "microsoft/layoutlmv3-base"
OUTPUT_DIR = "./layoutlmv3-sroie-model"

# Flat labels based on your JSON structure
LABELS = ["O", "COMPANY", "DATE", "ADDRESS", "TOTAL"]
LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for idx, label in enumerate(LABELS)}


# ==========================================
# 2. THE RAM-FRIENDLY DATA PARSER
# ==========================================
def normalize_bbox(box, width, height):
    return [
        max(0, min(1000, int(1000 * (box[0] / width)))),
        max(0, min(1000, int(1000 * (box[1] / height)))),
        max(0, min(1000, int(1000 * (box[2] / width)))),
        max(0, min(1000, int(1000 * (box[3] / height)))),
    ]

def load_sroie_split(split_name):
    split_dir = os.path.join(DATASET_DIR, split_name)
    img_dir, box_dir, ent_dir = [os.path.join(split_dir, f) for f in ["img", "box", "entities"]]

    examples = {"image_path": [], "words": [], "bboxes": [], "ner_tags": []}

    for filename in os.listdir(img_dir):
        if not filename.endswith(".jpg"): continue
        file_id = filename.split(".")[0]

        img_path = os.path.join(img_dir, filename)
        box_path = os.path.join(box_dir, file_id + ".txt")
        ent_path = os.path.join(ent_dir, file_id + ".txt")

        if not (os.path.exists(box_path) and os.path.exists(ent_path)): continue

        try:
            with Image.open(img_path) as img:
                width, height = img.size

            # ADDED: errors='ignore' to bypass weird characters like the Pound sign (£)
            with open(ent_path, 'r', encoding='utf-8', errors='ignore') as f:
                entities = json.load(f)
                # print(entities)
        except Exception:
            continue

        words, bboxes, ner_tags = [], [], []

        # ADDED: errors='ignore' here as well
        with open(box_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) < 9: continue

                coords = [int(p) for p in parts[:8]]
                text_fragment = ",".join(parts[8:]).strip()

                x_min = min(coords[0], coords[2], coords[4], coords[6])
                y_min = min(coords[1], coords[3], coords[5], coords[7])
                x_max = max(coords[0], coords[2], coords[4], coords[6])
                y_max = max(coords[1], coords[3], coords[5], coords[7])

                if x_max <= x_min or y_max <= y_min: continue

                assigned_label = "O"
                for key, full_string in entities.items():
                    if full_string and text_fragment in full_string:
                        assigned_label = key.upper()
                        break

                words.append(text_fragment)
                bboxes.append(normalize_bbox([x_min, y_min, x_max, y_max], width, height))
                ner_tags.append(LABEL2ID[assigned_label])

        examples["image_path"].append(img_path)
        examples["words"].append(words)
        examples["bboxes"].append(bboxes)
        examples["ner_tags"].append(ner_tags)

    return Dataset.from_dict(examples)


# ==========================================
# 3. INITIALIZATION & PREPROCESSING (OOM SAFE)
# ==========================================
print("Loading and parsing raw data...")
raw_datasets = DatasetDict({
    "train": load_sroie_split("train"),
    "test": load_sroie_split("test")
})

Loading and parsing raw data...


In [ ]:
raw_datasets['train']['image_path']

Column(['./sroie-datasetv2/SROIE2019/train/img/X51005757353.jpg', './sroie-datasetv2/SROIE2019/train/img/X51006556732.jpg', './sroie-datasetv2/SROIE2019/train/img/X51006619758.jpg', './sroie-datasetv2/SROIE2019/train/img/X51005676549.jpg', './sroie-datasetv2/SROIE2019/train/img/X51005742068.jpg'])

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class ReceiptDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = Image.open(item['image_path']).convert('RGB')
        label = item['label']  # adjust to your actual label column name

        if self.transform:
            image = self.transform(image)

        return image, label

data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.225])  # 1 channel after Grayscale
])

chars74k_dataset = ReceiptDataset(raw_datasets['train'], transform=data_transform)
dataset_loader = torch.utils.data.DataLoader(
    chars74k_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=4
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
class Binarize:
    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def __call__(self, tensor):
        return (tensor > self.threshold).float()

data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(),
    transforms.ToTensor(),           # scales pixels to [0, 1]
    Binarize(threshold=0.5),         # pixels → 0.0 or 1.0
    transforms.Normalize(mean=[0.5], std=[0.225])
])

In [ ]:
import cv2
import numpy as np

class OtsuBinarize:
    def __call__(self, tensor):
        # Convert tensor [1, H, W] → numpy uint8
        img_np = (tensor.squeeze(0).numpy() * 255).astype(np.uint8)
        _, binary = cv2.threshold(img_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return torch.from_numpy(binary / 255.0).unsqueeze(0).float()

data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(),
    transforms.ToTensor(),           # must come before OtsuBinarize
    OtsuBinarize(),
    transforms.Normalize(mean=[0.5], std=[0.225])
])

In [ ]:
print(raw_datasets['train'].features)
print(raw_datasets['train'][0])  # see first sample's keys

{'image_path': Value('string'), 'words': List(Value('string')), 'bboxes': List(List(Value('int64'))), 'ner_tags': List(Value('int64'))}
{'image_path': './sroie-datasetv2/SROIE2019/train/img/X51005757353.jpg', 'words': ['MR. D.I.Y. (M) SDN BHD', '(CO.REG :860671-D)', 'LOT 1851-A & 1851-B, JALAN KPB 6,', 'KAWASAN PERINDUSTRIAN BALAKONG,', '43300 SERI KEMBANGAN, SELANGOR', '(GST ID NO :000306020352)', '(IOI PUCHONG)', '-TAX INVOICE-', "WHITE CABLE TIE 4*200MM(8')", '*S', 'UB52 - 20/250', '9072317', '1 X 3.90', '3.90', 'WHITE CABLE TIE 5*250', '*S', 'UH32 - 10/150', '9072318', '1 X 6.50', '6.50', 'HOSE PUMP C88351#', '*S', 'KE23-33-53 - 12/120', '9074333', '1 X 1.90', '1.90', 'ITEM(S) : 3', 'QTY(S) : 3', 'TOTAL INCL. GST@6%', 'RM 12.30', 'CASH', 'RM 50.00', 'CHANGE', 'RM 37.70', 'GST @6% INCLUDED IN TOTAL', 'RM 0.70', '24-03-18 18:10 SH01 ZJ86', 'T1 R000112046', 'OPERATOR TRAINEE CASHIER', 'EXCHANGE ARE ALLOWED WITHIN', '7 DAY WITH RECEIPT .', 'STRICTLY NO CASH REFUND .'], 'bboxes': [[202,

In [ ]:
class ReceiptDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = Image.open(item['image_path']).convert('RGB')
        label = item['YOUR_ACTUAL_COLUMN']  # ← replace this

        if self.transform:
            image = self.transform(image)

        return image, label